# Deteksi Keretakan Permukaan Beton menggunakan CNN
**Mata Kuliah:** Advanced Computer Vision

**Metodologi:**
1. Data Collection (Kaggle API)
2. Data Preprocessing & Splitting
3. Modeling (Transfer Learning MobileNetV2)
4. Training
5. Evaluasi (Accuracy, Confusion Matrix, Recall, ROC-AUC)

In [ ]:
# 1. DATA COLLECTION VIA KAGGLE API
import os
from google.colab import files

print("Silakan upload file kaggle.json Anda:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("\nMengunduh dataset dari Kaggle...")
!kaggle datasets download -d arunrk7/surface-crack-detection

print("\nMengekstrak dataset...")
!unzip -q surface-crack-detection.zip -d dataset/

print("Dataset berhasil diunduh dan diekstrak ke folder 'dataset/'!")

In [ ]:
# 2. DATA PREPROCESSING & SPLITTING
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
import matplotlib.pyplot as plt
import numpy as np

BATCH_SIZE = 32
IMG_SIZE = (128, 128)

print("Memuat Data Latih (Training Set)...")
train_dataset = image_dataset_from_directory(
    'dataset/',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

print("\nMemuat Data Uji (Validation/Test Set)...")
validation_dataset = image_dataset_from_directory(
    'dataset/',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# 3. MODELING (Arsitektur MobileNetV2)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./127.5, offset=-1),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=['accuracy'])

model.summary()

In [ ]:
# 4. TRAINING
EPOCHS = 5
print(f"Memulai training selama {EPOCHS} epochs...")

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS
)

In [ ]:
# 5. EVALUATION & VISUALIZATION
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Akurasi Latih')
plt.plot(val_acc, label='Akurasi Uji')
plt.legend(loc='lower right')
plt.title('Grafik Akurasi')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Loss Latih')
plt.plot(val_loss, label='Loss Uji')
plt.legend(loc='upper right')
plt.title('Grafik Loss')
plt.show()

In [ ]:
# 6. CONFUSION MATRIX & CLASSIFICATION REPORT
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("Menghitung prediksi pada Data Uji...")
y_true = []
y_pred_probs = []

for images, labels in validation_dataset:
    y_true.extend(labels.numpy())
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds)

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs).flatten()
y_pred = (y_pred_probs > 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative (Aman)', 'Positive (Retak)'],
            yticklabels=['Negative (Aman)', 'Positive (Retak)'])
plt.title('Confusion Matrix')
plt.ylabel('Label Asli')
plt.xlabel('Label Prediksi')
plt.show()

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_true, y_pred, target_names=['Negative (Aman)', 'Positive (Retak)']))

In [ ]:
# 7. ROC CURVE & AUC
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Skor AUC: {roc_auc:.3f}")